# DPO-5: MT-Bench — Mistral Base + SFT Checkpoint

Anchors rows 1 and 2 of the money table with real MT-Bench scores.

| Run ID | Model | Expected |
|---|---|---|
| `dpo5_base` | Mistral-7B-v0.1 (base) | ~3–4 |
| `dpo5_sft` | SFT QLoRA checkpoint-17205 | ~6–6.5 |

**Cost:** ~\$10–20 total (GPT-4 judge, 160 turns × 2 runs)  
**Time:** ~45–60 min total (~15 min gen + ~10 min API per run)  
**Pipeline:** `gen_model_answer` (GPU, free) → `gen_judgment` (GPT-4) → parse `gpt-4_single.jsonl` → `results/runs.csv`

In [ ]:
import sys, os, json, subprocess, shutil, tempfile, csv, statistics
from pathlib import Path

REPO_ROOT      = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

BASE_MODEL     = "mistralai/Mistral-7B-v0.1"
SFT_CHECKPOINT = str(REPO_ROOT / "checkpoints/sft-zephyr-lora/checkpoint-17205")
RUNS_CSV       = REPO_ROOT / "results" / "runs.csv"

# FastChat model IDs — used as filenames inside llm_judge/data/
BASE_MODEL_ID = "mistral-base"
SFT_MODEL_ID  = "sft-qlora"

# run_id values written to runs.csv
BASE_RUN_ID = "dpo5_base"
SFT_RUN_ID  = "dpo5_sft"

print(f"SFT checkpoint exists : {Path(SFT_CHECKPOINT).exists()}")
print(f"runs.csv              : {RUNS_CSV}")

In [ ]:
# Set OPENAI_API_KEY in your shell before launching Jupyter.
# PowerShell: $env:OPENAI_API_KEY = 'sk-...'
# Do NOT hardcode your key in this file.
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY not set.\n"
    "PowerShell: $env:OPENAI_API_KEY = 'sk-...'"
)
print("OPENAI_API_KEY set ✓")

## 0. Install fschat (once)

In [ ]:
result = subprocess.run(
    [sys.executable, "-c", "import fastchat; print(fastchat.__version__)"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("Installing fschat...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fschat[model_worker,llm_judge]"],
        check=True,
    )
    import importlib; importlib.invalidate_caches()
else:
    print(f"fschat {result.stdout.strip()} already installed ✓")

import fastchat
JUDGE_DIR = Path(fastchat.__file__).parent / "llm_judge"
assert JUDGE_DIR.is_dir(), f"llm_judge dir not found: {JUDGE_DIR}"
print(f"llm_judge dir: {JUDGE_DIR}")

In [ ]:

# FastChat pip package omits the llm_judge data files — download them from GitHub.
import urllib.request

FASTCHAT_RAW = "https://raw.githubusercontent.com/lm-sys/FastChat/main/fastchat/llm_judge"

DATA_FILES = [
    "data/mt_bench/question.jsonl",
    "data/mt_bench/reference_answer/gpt-4.jsonl",
    "data/judge_prompts.jsonl",
]

for rel in DATA_FILES:
    dest = JUDGE_DIR / rel
    if dest.exists():
        print(f"  exists  {rel}")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{FASTCHAT_RAW}/{rel}"
    print(f"  fetch   {rel} ...", end=" ", flush=True)
    urllib.request.urlretrieve(url, dest)
    print("✓")

print("\nMT-Bench data files ready ✓")


## Helpers

In [ ]:

CSV_FIELDNAMES = [
    "run_id", "checkpoint", "tag", "stage",
    "beta", "epochs", "lr", "lora_r", "simpo_gamma",
    "avg_gen_length", "p90_gen_length",
    "harmful_refusal_rate", "over_refusal_rate", "pref_acc",
    "mt_bench", "alpacaeval2_lc", "notes",
]

def stream_run(cmd, cwd=None, auto_confirm=False):
    """Run subprocess, forward env, stream output. auto_confirm sends Enter to any stdin prompts."""
    proc = subprocess.Popen(
        cmd,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=str(cwd) if cwd else None,
        env=os.environ,
    )
    if auto_confirm:
        proc.stdin.write("\n")
        proc.stdin.flush()
        proc.stdin.close()
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")


def parse_score(model_id, judge_dir):
    """Average MT-Bench score from gpt-4_single.jsonl for a given model_id."""
    jf = Path(judge_dir) / "data" / "mt_bench" / "model_judgment" / "gpt-4_single.jsonl"
    if not jf.exists():
        raise FileNotFoundError(f"Judgment file not found: {jf}")
    scores = []
    with open(jf) as f:
        for line in f:
            if not line.strip():
                continue
            e = json.loads(line)
            if e.get("model") != model_id:
                continue
            s = e.get("score", -1)
            if isinstance(s, (int, float)) and s != -1:
                scores.append(float(s))
    if not scores:
        raise ValueError(f"No scores found for {model_id!r} in {jf}")
    avg = round(statistics.mean(scores), 2)
    print(f"  {model_id}: {avg:.2f}  ({len(scores)} turns scored)")
    return avg


def append_csv(row: dict):
    write_header = not RUNS_CSV.exists() or RUNS_CSV.stat().st_size == 0
    with open(RUNS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  Appended to {RUNS_CSV}")


print("Helpers defined ✓")


## 1. MT-Bench — Mistral-7B Base

No LoRA — loads `mistralai/Mistral-7B-v0.1` directly.  
Expected score: **~3–4**.

In [ ]:
# ~12–15 min on RTX 4090
print(f"[1/2] gen_model_answer: {BASE_MODEL_ID}")
stream_run(
    [
        sys.executable, "-m", "fastchat.llm_judge.gen_model_answer",
        "--model-path", BASE_MODEL,
        "--model-id",   BASE_MODEL_ID,
        "--bench-name", "mt_bench",
        "--num-gpus-per-model", "1",
    ],
    cwd=JUDGE_DIR,
)
print("Done ✓")

In [ ]:

# ~$5–10, ~10 min — GPT-4 judges all 160 turns
print(f"[2/2] gen_judgment: {BASE_MODEL_ID}  (~$5–10)")
stream_run(
    [
        sys.executable, "-m", "fastchat.llm_judge.gen_judgment",
        "--model-list",  BASE_MODEL_ID,
        "--judge-model", "gpt-4",
        "--bench-name",  "mt_bench",
        "--mode",        "single",
    ],
    cwd=JUDGE_DIR,
    auto_confirm=True,
)
print("Done ✓")


In [11]:

# Check gen_judgment progress while it's still running.
judgment_file = JUDGE_DIR / "data" / "mt_bench" / "model_judgment" / "gpt-4_single.jsonl"

if not judgment_file.exists():
    print("Judgment file not created yet — still starting up.")
else:
    entries = [json.loads(l) for l in judgment_file.read_text().splitlines() if l.strip()]
    model_entries = [e for e in entries if e.get("model") == BASE_MODEL_ID]
    print(f"Judged so far: {len(model_entries)} / 160 turns")
    if model_entries:
        last = model_entries[-1]
        print(f"  Last: Q{last['question_id']} turn {last.get('turn')}  score={last.get('score')}")


Judged so far: 160 / 160 turns
  Last: Q130 turn 2  score=2


In [12]:
base_score = parse_score(BASE_MODEL_ID, JUDGE_DIR)

append_csv({
    "run_id":     BASE_RUN_ID,
    "checkpoint": BASE_MODEL,
    "tag":        "base",
    "stage":      "base",
    "mt_bench":   base_score,
    "notes":      "RTX 4090, Mistral-7B-v0.1 float16",
})

print(f"\nMistral base MT-Bench: {base_score:.2f}  (target ~3–4)")

  mistral-base: 3.17  (156 turns scored)
  Appended to D:\git\DPOTuning\results\runs.csv

Mistral base MT-Bench: 3.17  (target ~3–4)


## 2. MT-Bench — SFT QLoRA Checkpoint

Merges LoRA adapter into float16 base → FastChat → clean up temp dir.  
Expected score: **~6–6.5**.

In [13]:
# ~5–8 min on CPU (float16, ~14 GB RAM)
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged_dir = Path(tempfile.mkdtemp(prefix="mt_bench_sft_"))
print(f"Merging LoRA → {merged_dir} ...")

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cpu"
)
model = PeftModel.from_pretrained(base, SFT_CHECKPOINT)
model = model.merge_and_unload()
model.save_pretrained(str(merged_dir))
AutoTokenizer.from_pretrained(SFT_CHECKPOINT).save_pretrained(str(merged_dir))

del model, base
torch.cuda.empty_cache()
print(f"Merged ✓  →  {merged_dir}")

`torch_dtype` is deprecated! Use `dtype` instead!


Merging LoRA → C:\Users\bluebyte\AppData\Local\Temp\mt_bench_sft_5asm5yph ...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

c:\Users\bluebyte\miniconda3\envs\finetune\Lib\site-packages\peft\config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged ✓  →  C:\Users\bluebyte\AppData\Local\Temp\mt_bench_sft_5asm5yph


In [14]:
print(f"[1/2] gen_model_answer: {SFT_MODEL_ID}")
stream_run(
    [
        sys.executable, "-m", "fastchat.llm_judge.gen_model_answer",
        "--model-path", str(merged_dir),
        "--model-id",   SFT_MODEL_ID,
        "--bench-name", "mt_bench",
        "--num-gpus-per-model", "1",
    ],
    cwd=JUDGE_DIR,
)
print("Done ✓")

[1/2] gen_model_answer: sft-qlora
Output to data/mt_bench/model_answer/sft-qlora.jsonl
`torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 5388.92it/s]

  0%|          | 0/80 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  1%|▏         | 1/80 [00:09<1

In [15]:

print(f"[2/2] gen_judgment: {SFT_MODEL_ID}  (~$5–10)")
stream_run(
    [
        sys.executable, "-m", "fastchat.llm_judge.gen_judgment",
        "--model-list",  SFT_MODEL_ID,
        "--judge-model", "gpt-4",
        "--bench-name",  "mt_bench",
        "--mode",        "single",
    ],
    cwd=JUDGE_DIR,
    auto_confirm=True,
)
print("Done ✓")


[2/2] gen_judgment: sft-qlora  (~$5–10)
Stats:
{
    "bench_name": "mt_bench",
    "mode": "single",
    "judge": "gpt-4",
    "baseline": null,
    "model_list": [
        "sft-qlora"
    ],
    "total_num_questions": 80,
    "total_num_matches": 160,
    "output_path": "data/mt_bench/model_judgment/gpt-4_single.jsonl"
}
Press Enter to confirm...
  0%|          | 0/160 [00:00<?, ?it/s]question: 81, turn: 1, model: sft-qlora, score: 9, judge: ('gpt-4', 'single-v1')

  1%|          | 1/160 [00:05<13:52,  5.24s/it]question: 82, turn: 1, model: sft-qlora, score: 10, judge: ('gpt-4', 'single-v1')

  1%|▏         | 2/160 [00:10<13:28,  5.12s/it]question: 83, turn: 1, model: sft-qlora, score: 10, judge: ('gpt-4', 'single-v1')

  2%|▏         | 3/160 [00:14<12:05,  4.62s/it]question: 84, turn: 1, model: sft-qlora, score: 8, judge: ('gpt-4', 'single-v1')

  2%|▎         | 4/160 [00:18<11:55,  4.59s/it]question: 85, turn: 1, model: sft-qlora, score: 9, judge: ('gpt-4', 'single-v1')

  3%|▎     

In [16]:
sft_score = parse_score(SFT_MODEL_ID, JUDGE_DIR)

append_csv({
    "run_id":     SFT_RUN_ID,
    "checkpoint": SFT_CHECKPOINT,
    "tag":        "sft",
    "stage":      "sft",
    "epochs":     "1",
    "lr":         "2e-4",
    "lora_r":     "16",
    "mt_bench":   sft_score,
    "notes":      "RTX 4090, UltraChat 1 epoch QLoRA",
})

# Free the ~28 GB temp merged model
shutil.rmtree(merged_dir, ignore_errors=True)
print(f"Temp dir cleaned up ✓")
print(f"\nSFT QLoRA MT-Bench: {sft_score:.2f}  (target ~6–6.5)")

  sft-qlora: 5.75  (160 turns scored)
  Appended to D:\git\DPOTuning\results\runs.csv
Temp dir cleaned up ✓

SFT QLoRA MT-Bench: 5.75  (target ~6–6.5)


## 3. Results

Compare against Zephyr published baseline.

In [17]:
rows = [
    ("Mistral-7B-v0.1 base",          base_score, "~3–4"),
    ("SFT QLoRA (checkpoint-17205)",   sft_score,  "~6–6.5"),
    ("Zephyr-7B-β (published, full-FT)", 7.34,     "ref"),
]

print("=" * 58)
print("MT-Bench Results — DPO-5")
print("=" * 58)
print(f"{'Model':<38} {'Score':>6}  {'Target':>8}")
print("-" * 58)
for label, score, target in rows:
    print(f"{label:<38} {score:>6.2f}  {target:>8}")
print("=" * 58)
print(f"\nSFT alignment effect  : +{sft_score - base_score:.2f} MT-Bench points")
print(f"QLoRA gap vs Zephyr   : {7.34 - sft_score:.2f} points  (target: ~0.5–1.0 for QLoRA)")

MT-Bench Results — DPO-5
Model                                   Score    Target
----------------------------------------------------------
Mistral-7B-v0.1 base                     3.17      ~3–4
SFT QLoRA (checkpoint-17205)             5.75    ~6–6.5
Zephyr-7B-β (published, full-FT)         7.34       ref

SFT alignment effect  : +2.58 MT-Bench points
QLoRA gap vs Zephyr   : 1.59 points  (target: ~0.5–1.0 for QLoRA)
